# Selected corrupted-beam comparison for DVL-DeepONet

This notebook compares the requested missing-beam methods:

1. ELC
2. MissBeamNet
3. DVL-DeepONet averaged IMU (MSE only)
4. ours raw 100 Hz IMU branch (NAViLoss)


The neural models train with corrupted DVL beams and are evaluated on full-beam and missing-beam scenarios.


In [1]:
# ============================================================
# 0. Imports, setup, and reproducibility #set 1,2,3
# ============================================================

import json
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader




SEED = 1234




def set_seed(seed=SEED, deterministic=False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.backends.cudnn.benchmark = torch.cuda.is_available()

set_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
PIN_MEMORY = DEVICE.type == "cuda"
print("Using device:", DEVICE)

SAVE_DIR = Path("dvl_deeponet_100_imu_missing_beam_naviloss")
FIG_DIR = SAVE_DIR / "figures"
CKPT_DIR = SAVE_DIR / "checkpoints"
DATA_DIR = SAVE_DIR / "arrays"
for directory in (FIG_DIR, CKPT_DIR, DATA_DIR):
    directory.mkdir(parents=True, exist_ok=True)

print("Results folder:", SAVE_DIR.resolve())


Using device: cuda
Results folder: /home/asahoo/navonet/dvl_deeponet_100_imu_missing_beam_naviloss


In [2]:
# ============================================================
# 1. Load A-KIT DVL beams, high-rate IMU, and Cartesian velocity GT
# ============================================================
# Expected per trajectory:
#   A-KIT/Data/Trajectory1/DVL_beams_traj_1.npy
#   A-KIT/Data/Trajectory1/DVL_trajectory1.csv
#   A-KIT/AUV_IMU_npy/IMU_trajectory1.npy
#
# DVL_beams_traj_*.npy shape:
#   (N,5) = [time, beam1, beam2, beam3, beam4]
#   or (N,4) = [beam1, beam2, beam3, beam4]
#
# IMU_trajectory*.npy shape:
#   (M,7) = [time, ax, ay, az, gx, gy, gz]
#   usually 100 Hz, resampled here to DVL timestamps.
# ============================================================

import re
import pandas as pd

DATA_ROOT = Path("A-KIT/Data")
IMU_ROOT = Path("A-KIT/AUV_IMU_npy")
ALPHA_DEG = 20.0


TRAIN_TRAJECTORIES = None
VAL_TRAJECTORIES = None
TEST_TRAJECTORIES = None

REQUIRED_VELOCITY_COLUMNS = ["DVL X [m/s]", "DVL Y [m/s]", "DVL Z [m/s]"]

# IMU_trajectory*.npy format is fixed by your data:
# [time, ax, ay, az, gx, gy, gz]
IMU_FEATURE_NAMES = ["ax", "ay", "az", "gx", "gy", "gz"]
IMU_RESAMPLE_MODE = "average"



def build_dvl_H(alpha_deg=20.0):
    """Build the 4x3 DVL beam geometry matrix used by the A-KIT beam generator."""
    alpha = np.deg2rad(alpha_deg)
    H = []
    for i in range(4):
        psi = np.deg2rad(45 + i * 90)
        H.append([
            np.cos(psi) * np.sin(alpha),
            np.sin(psi) * np.sin(alpha),
            np.cos(alpha),
        ])
    return np.asarray(H, dtype=np.float32)


def natural_traj_number(path):
    s = str(path)
    m = re.search(r"Trajectory(\d+)", s)
    if m:
        return int(m.group(1))
    m = re.search(r"trajectory(\d+)", s, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    m = re.search(r"traj_(\d+)", s)
    if m:
        return int(m.group(1))
    return 10**9


def find_imu_path(traj_num):
    candidates = [
        IMU_ROOT / f"IMU_trajectory{traj_num}.npy",
        IMU_ROOT / f"IMU_trajectory_{traj_num}.npy",
        IMU_ROOT / f"imu_trajectory{traj_num}.npy",
        IMU_ROOT / f"imu_trajectory_{traj_num}.npy",
    ]
    for path in candidates:
        if path.exists():
            return path
    matches = sorted(IMU_ROOT.glob(f"*trajectory*{traj_num}*.npy"), key=natural_traj_number)
    if matches:
        return matches[0]
    raise FileNotFoundError(f"No IMU file found for trajectory {traj_num} inside {IMU_ROOT.resolve()}")


def load_and_resample_imu(traj_num, dvl_time_local):
    imu_path = find_imu_path(traj_num)
    imu = np.load(imu_path).astype(np.float32)
    if imu.ndim != 2 or imu.shape[1] != 7:
        raise ValueError(f"{imu_path} must have shape (M,7) = [time, ax, ay, az, gx, gy, gz], got {imu.shape}")

    imu_time = imu[:, 0]
    imu_values = imu[:, 1:7]
    if not np.isfinite(imu).all():
        raise ValueError(f"Non-finite values found in {imu_path}")
    if np.any(np.diff(imu_time) <= 0):
        raise ValueError(f"IMU time must be strictly increasing in {imu_path}")

    dvl_time_local = np.asarray(dvl_time_local, dtype=np.float32).reshape(-1)

    # Align if both sensors start from different absolute offsets but share the same relative clock.
    imu_time_rel = imu_time - imu_time[0]
    dvl_time_rel = dvl_time_local - dvl_time_local[0]

    if dvl_time_rel[0] < imu_time_rel[0] - 1e-6 or dvl_time_rel[-1] > imu_time_rel[-1] + 1e-6:
        print(
            f"Warning Trajectory{traj_num}: DVL time [{dvl_time_rel[0]:.3f}, {dvl_time_rel[-1]:.3f}] "
            f"extends beyond IMU time [{imu_time_rel[0]:.3f}, {imu_time_rel[-1]:.3f}]. Edge IMU values will be held."
        )

    if IMU_RESAMPLE_MODE != "average":
        raise ValueError(f"Unsupported IMU_RESAMPLE_MODE: {IMU_RESAMPLE_MODE}")
    
    imu_resampled = np.zeros((len(dvl_time_rel), imu_values.shape[1]), dtype=np.float32)
    
    for k in range(len(dvl_time_rel)):
    
        if k == 0:
            t_start = dvl_time_rel[0]
            t_end = dvl_time_rel[1]
        else:
            t_start = dvl_time_rel[k - 1]
            t_end = dvl_time_rel[k]
    
        mask = (imu_time_rel >= t_start) & (imu_time_rel < t_end)
    
        if np.any(mask):
            imu_resampled[k, :] = np.mean(imu_values[mask, :], axis=0)
        else:
            # fallback: use nearest IMU sample if no IMU sample is found in interval
            nearest_idx = np.argmin(np.abs(imu_time_rel - dvl_time_rel[k]))
            imu_resampled[k, :] = imu_values[nearest_idx, :]
    
   

    if len(imu_time) > 1:
        imu_dt = float(np.mean(np.diff(imu_time)))
        imu_fs = 1.0 / imu_dt if imu_dt > 0 else float("nan")
    else:
        imu_fs = float("nan")

    return imu_resampled, imu_path, imu_fs


def load_akit_dvl_imu_velocity_data(data_root=DATA_ROOT, alpha_deg=20.0):
    data_root = Path(data_root)
    H_np = build_dvl_H(alpha_deg=alpha_deg)
    traj_dirs = sorted(data_root.glob("Trajectory*"), key=natural_traj_number)
    if len(traj_dirs) == 0:
        raise FileNotFoundError(f"No Trajectory* folders found inside: {data_root.resolve()}")
    if not IMU_ROOT.exists():
        raise FileNotFoundError(f"IMU_ROOT does not exist: {IMU_ROOT.resolve()}")

    all_t, all_beams, all_imu, all_vel, all_traj_id, summaries = [], [], [], [], [], []
    global_time_offset = 0.0

    for traj_dir in traj_dirs:
        traj_num = natural_traj_number(traj_dir)
        beam_path = traj_dir / f"DVL_beams_traj_{traj_num}.npy"
        csv_path = traj_dir / f"DVL_trajectory{traj_num}.csv"

        if not beam_path.exists() or not csv_path.exists():
            missing = [path.name for path in (beam_path, csv_path) if not path.exists()]
            print(f"Skipping Trajectory{traj_num}: missing {missing}")
            continue

        beams_npy = np.load(beam_path).astype(np.float32)
        df = pd.read_csv(csv_path)
        missing_cols = [col for col in REQUIRED_VELOCITY_COLUMNS if col not in df.columns]
        if missing_cols:
            raise ValueError(f"{csv_path} missing required columns: {missing_cols}")

        if beams_npy.ndim != 2 or beams_npy.shape[1] not in (4, 5):
            raise ValueError(f"{beam_path} must have shape (N,4) or (N,5), got {beams_npy.shape}")

        if beams_npy.shape[1] == 5:
            t_local = beams_npy[:, 0:1]
            beams = beams_npy[:, 1:5]
        else:
            beams = beams_npy[:, 0:4]
            if "Time [s]" in df.columns:
                t_local = df["Time [s]"].to_numpy(dtype=np.float32).reshape(-1, 1)
            else:
                t_local = np.arange(len(beams), dtype=np.float32).reshape(-1, 1)

        vel = df[REQUIRED_VELOCITY_COLUMNS].to_numpy(dtype=np.float32)
        N = min(len(t_local), len(beams), len(vel))
        if N < 2:
            print(f"Skipping Trajectory{traj_num}: fewer than two aligned DVL samples")
            continue

        t_local = t_local[:N]
        beams = beams[:N]
        vel = vel[:N]

        imu_resampled, imu_path, imu_fs = load_and_resample_imu(traj_num, t_local[:, 0])
        imu_resampled = imu_resampled[:N]

        if not np.isfinite(t_local).all() or not np.isfinite(beams).all() or not np.isfinite(vel).all() or not np.isfinite(imu_resampled).all():
            raise ValueError(f"Non-finite values found in Trajectory{traj_num}")

        dt = np.diff(t_local[:, 0])
        if np.any(dt <= 0):
            raise ValueError(f"Local DVL time must be strictly increasing in Trajectory{traj_num}")
        dt_est = float(np.median(dt))

        t_shifted = t_local - float(t_local[0, 0]) + global_time_offset
        global_time_offset = float(t_shifted[-1, 0] + dt_est)

        all_t.append(t_shifted.astype(np.float32))
        all_beams.append(beams.astype(np.float32))
        all_imu.append(imu_resampled.astype(np.float32))
        all_vel.append(vel.astype(np.float32))
        all_traj_id.append(np.full((N, 1), traj_num, dtype=np.int64))
        summaries.append((traj_num, N, dt_est, str(imu_path), imu_fs))
        print(f"Loaded Trajectory{traj_num}: DVL N={N}, DVL dt~{dt_est:.4g}s, IMU fs~{imu_fs:.2f}Hz")

    if len(all_t) == 0:
        raise FileNotFoundError("No valid trajectory pairs were loaded. Check the A-KIT output paths.")

    t = np.vstack(all_t).astype(np.float32)
    dvl_beams = np.vstack(all_beams).astype(np.float32)
    imu_data = np.vstack(all_imu).astype(np.float32)
    dvl_cart_gt = np.vstack(all_vel).astype(np.float32)
    traj_id = np.vstack(all_traj_id).astype(np.int64)
    return t, dvl_beams, imu_data, dvl_cart_gt, H_np, traj_id, summaries


def validate_arrays(t, dvl_beams, imu_data, dvl_cart_gt, H_np, traj_id):
    t = np.asarray(t, dtype=np.float32)
    if t.ndim == 1:
        t = t[:, None]
    dvl_beams = np.asarray(dvl_beams, dtype=np.float32)
    imu_data = np.asarray(imu_data, dtype=np.float32)
    dvl_cart_gt = np.asarray(dvl_cart_gt, dtype=np.float32)
    H_np = np.asarray(H_np, dtype=np.float32)
    traj_id = np.asarray(traj_id, dtype=np.int64)
    if traj_id.ndim == 1:
        traj_id = traj_id[:, None]

    if t.ndim != 2 or t.shape[1] != 1:
        raise ValueError(f"time must have shape (N,) or (N,1); got {t.shape}")
    if dvl_beams.ndim != 2 or dvl_beams.shape[1] != 4:
        raise ValueError(f"dvl_beams must have shape (N,4); got {dvl_beams.shape}")
    if imu_data.ndim != 2 or imu_data.shape[1] != 6:
        raise ValueError(f"imu_data must have shape (N,6) = accel xyz + gyro xyz; got {imu_data.shape}")
    if dvl_cart_gt.ndim != 2 or dvl_cart_gt.shape[1] != 3:
        raise ValueError(f"dvl_cartesian_gt must have shape (N,3); got {dvl_cart_gt.shape}")
    if H_np.shape != (4, 3):
        raise ValueError(f"H_dvl must have shape (4,3); got {H_np.shape}")
    if not (len(t) == len(dvl_beams) == len(imu_data) == len(dvl_cart_gt) == len(traj_id)):
        raise ValueError("N mismatch between time, beams, IMU, Cartesian velocity, and trajectory IDs")
    if np.any(np.diff(t[:, 0]) <= 0):
        raise ValueError("global time values must be strictly increasing")

    arrays = {"time": t, "dvl_beams": dvl_beams, "imu_data": imu_data, "dvl_cartesian_gt": dvl_cart_gt, "H": H_np}
    bad = [name for name, arr in arrays.items() if not np.isfinite(arr).all()]
    if bad:
        raise ValueError(f"Non-finite values found in: {bad}")
    return t, dvl_beams, imu_data, dvl_cart_gt, H_np, traj_id


t, dvl_beams, imu_data, dvl_cart_gt, H_np, traj_id, traj_summaries = load_akit_dvl_imu_velocity_data(DATA_ROOT, ALPHA_DEG)
t, dvl_beams, imu_data, dvl_cart_gt, H_np, traj_id = validate_arrays(t, dvl_beams, imu_data, dvl_cart_gt, H_np, traj_id)
VEL_DIM = dvl_cart_gt.shape[1]
INPUT_DIM = dvl_beams.shape[1] + imu_data.shape[1]

DATA_DIR.mkdir(parents=True, exist_ok=True)
np.save(DATA_DIR / "akit_time_global.npy", t)
np.save(DATA_DIR / "akit_dvl_beams.npy", dvl_beams)
np.save(DATA_DIR / "akit_imu_resampled_to_dvl.npy", imu_data)
np.save(DATA_DIR / "akit_dvl_cartesian_gt.npy", dvl_cart_gt)
np.save(DATA_DIR / "akit_H_dvl.npy", H_np)
np.save(DATA_DIR / "akit_traj_id.npy", traj_id)

print("\nFinal connected dataset")
print("time shape            :", t.shape)
print("dvl_beams shape       :", dvl_beams.shape)
print("imu_data shape        :", imu_data.shape, IMU_FEATURE_NAMES)
print("dvl_cartesian_gt shape:", dvl_cart_gt.shape)
print("H shape               :", H_np.shape)
print("VEL_DIM               :", VEL_DIM)
print("INPUT_DIM             :", INPUT_DIM, "= 4 DVL beams + 6 interpolated IMU values")
print("Trajectory IDs        :", np.unique(traj_id[:, 0]).astype(int))


Loaded Trajectory1: DVL N=400, DVL dt~1.003s, IMU fs~100.00Hz
Loaded Trajectory2: DVL N=400, DVL dt~1.003s, IMU fs~100.00Hz
Loaded Trajectory3: DVL N=400, DVL dt~1.003s, IMU fs~100.00Hz
Loaded Trajectory4: DVL N=400, DVL dt~1.003s, IMU fs~100.00Hz
Loaded Trajectory5: DVL N=400, DVL dt~1.003s, IMU fs~100.00Hz
Loaded Trajectory6: DVL N=400, DVL dt~1.003s, IMU fs~100.00Hz
Loaded Trajectory7: DVL N=400, DVL dt~1.003s, IMU fs~100.00Hz
Loaded Trajectory8: DVL N=400, DVL dt~1.003s, IMU fs~100.00Hz
Loaded Trajectory9: DVL N=400, DVL dt~1.003s, IMU fs~100.00Hz
Loaded Trajectory10: DVL N=400, DVL dt~1.003s, IMU fs~100.00Hz
Loaded Trajectory11: DVL N=400, DVL dt~1.003s, IMU fs~100.00Hz
Loaded Trajectory12: DVL N=400, DVL dt~1.003s, IMU fs~100.00Hz
Loaded Trajectory13: DVL N=400, DVL dt~1.003s, IMU fs~100.00Hz

Final connected dataset
time shape            : (5200, 1)
dvl_beams shape       : (5200, 4)
imu_data shape        : (5200, 6) ['ax', 'ay', 'az', 'gx', 'gy', 'gz']
dvl_cartesian_gt shape: (5

In [3]:
# ============================================================
# 2. Normalization, trajectory-aware split, and LS baseline
# ============================================================

WINDOW = 2
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15

# Fixed trajectory-level train/val/test split. 
USE_FIVE_FOLD_CV = True
CV_FOLD = 1  
N_CV_FOLDS = 5
# TRAIN_TRAJECTORIES =[1, 6, 7, 8, 9, 10, 11, 12, 13]                 
# VAL_TRAJECTORIES = [2,3] 
# TEST_TRAJECTORIES = [4,5]

def safe_std(x, axis=0, eps=1e-8):
    return np.maximum(np.std(x, axis=axis, keepdims=True), eps).astype(np.float32)


def build_kfold_cv_splits(unique_ids, train_ratio=0.70, val_ratio=0.15, n_folds=5):
    unique_ids = np.asarray(sorted(unique_ids), dtype=np.int64)
    n_total = len(unique_ids)
    if n_total < 6:
        raise ValueError("Need at least six trajectories for useful k-fold train/val/test splits")

    n_train = max(1, int(round(n_total * train_ratio)))
    n_val = max(1, int(round(n_total * val_ratio)))
    if n_train + n_val >= n_total:
        n_train = max(1, n_total - 2)
        n_val = 1

    splits = []
    for fold in range(n_folds):
        offset = int(round(fold * n_total / n_folds))
        rolled = np.roll(unique_ids, -offset)
        train_ids = np.asarray(sorted(rolled[:n_train]), dtype=np.int64)
        val_ids = np.asarray(sorted(rolled[n_train:n_train + n_val]), dtype=np.int64)
        test_ids = np.asarray(sorted(rolled[n_train + n_val:]), dtype=np.int64)
        splits.append({"train": train_ids, "val": val_ids, "test": test_ids})
    return splits


def split_trajectory_ids(unique_ids, train_ratio=0.70, val_ratio=0.15):
    unique_ids = np.asarray(sorted(unique_ids), dtype=np.int64)

    # Manual lists override the automatic CV split.
    if TRAIN_TRAJECTORIES is not None or VAL_TRAJECTORIES is not None or TEST_TRAJECTORIES is not None:
        train_ids = np.asarray([] if TRAIN_TRAJECTORIES is None else TRAIN_TRAJECTORIES, dtype=np.int64)
        val_ids = np.asarray([] if VAL_TRAJECTORIES is None else VAL_TRAJECTORIES, dtype=np.int64)
        test_ids = np.asarray([] if TEST_TRAJECTORIES is None else TEST_TRAJECTORIES, dtype=np.int64)
        cv_splits = None
    elif USE_FIVE_FOLD_CV:
        cv_splits = build_kfold_cv_splits(unique_ids, train_ratio, val_ratio, N_CV_FOLDS)
        if CV_FOLD < 0 or CV_FOLD >= len(cv_splits):
            raise ValueError(f"CV_FOLD must be one of 0, 1, 2, 3, 4; got {CV_FOLD}")
        selected = cv_splits[CV_FOLD]
        train_ids = selected["train"]
        val_ids = selected["val"]
        test_ids = selected["test"]
    else:
        cv_splits = None
        n_total = len(unique_ids)
        if n_total < 3:
            raise ValueError("Need at least three trajectories for trajectory-level train/val/test split")
        n_train = max(1, int(round(n_total * train_ratio)))
        n_val = max(1, int(round(n_total * val_ratio)))
        if n_train + n_val >= n_total:
            n_train = max(1, n_total - 2)
            n_val = 1
        train_ids = unique_ids[:n_train]
        val_ids = unique_ids[n_train:n_train + n_val]
        test_ids = unique_ids[n_train + n_val:]

    assigned = np.concatenate([train_ids, val_ids, test_ids])
    if len(assigned) != len(np.unique(assigned)):
        raise ValueError("Train/val/test trajectory lists overlap")
    missing = set(unique_ids.tolist()) - set(assigned.tolist())
    if missing:
        print("Unused trajectory IDs:", sorted(missing))
    if len(train_ids) == 0 or len(val_ids) == 0 or len(test_ids) == 0:
        raise ValueError("Train, validation, and test trajectory sets must all be non-empty")
    return train_ids, val_ids, test_ids, cv_splits


def indices_for_trajectories(ids):
    ids = np.asarray(ids, dtype=np.int64)
    return np.flatnonzero(np.isin(traj_id[:, 0], ids))


def beams_to_cartesian_ls(beams, H, vel_dim):
    v3 = beams @ np.linalg.pinv(H).T
    return v3[:, :vel_dim].astype(np.float32)


N = len(t)
unique_traj_ids = np.unique(traj_id[:, 0])
train_traj_ids, val_traj_ids, test_traj_ids, cv_splits = split_trajectory_ids(unique_traj_ids, TRAIN_RATIO, VAL_RATIO)
idx_train = indices_for_trajectories(train_traj_ids)
idx_val = indices_for_trajectories(val_traj_ids)
idx_test = indices_for_trajectories(test_traj_ids)

dvl_cart_ls = beams_to_cartesian_ls(dvl_beams, H_np, VEL_DIM)
raw_features = np.concatenate([dvl_beams, imu_data], axis=1).astype(np.float32)

# Compute normalization statistics from training samples only.
t_mean = t[idx_train].mean(axis=0, keepdims=True).astype(np.float32)
t_std = safe_std(t[idx_train], axis=0)
feature_mean = raw_features[idx_train].mean(axis=0, keepdims=True).astype(np.float32)
feature_std = safe_std(raw_features[idx_train], axis=0)
vel_mean = dvl_cart_gt[idx_train].mean(axis=0, keepdims=True).astype(np.float32)
vel_std = safe_std(dvl_cart_gt[idx_train], axis=0)

t_n = (t - t_mean) / t_std
features_n = (raw_features - feature_mean) / feature_std
vel_gt_n = (dvl_cart_gt - vel_mean) / vel_std

if cv_splits is not None:
    print("Available 5-fold trajectory splits:")
    for fold, split in enumerate(cv_splits):
        marker = " <== active" if fold == CV_FOLD else ""
        print(
            f"Fold {fold}: "
            f"train={split['train'].tolist()} | "
            f"val={split['val'].tolist()} | "
            f"test={split['test'].tolist()}" + marker
        )

print("Train trajectory IDs:", train_traj_ids)
print("Val trajectory IDs  :", val_traj_ids)
print("Test trajectory IDs :", test_traj_ids)
print("Train samples:", len(idx_train))
print("Val samples  :", len(idx_val))
print("Test samples :", len(idx_test))
print("Interpolated dvl_deeponet feature shape:", raw_features.shape)
print("Feature mean shape:", feature_mean.shape)
print("Feature std shape :", feature_std.shape)
print("Velocity mean:", vel_mean)
print("Velocity std :", vel_std)

ls_err = np.linalg.norm(dvl_cart_ls - dvl_cart_gt[:, :VEL_DIM], axis=1)
print("LS sanity RMSE [m/s]:", float(np.sqrt(np.mean(ls_err ** 2))))
print("LS sanity mean error [m/s]:", float(np.mean(ls_err)))


Available 5-fold trajectory splits:
Fold 0: train=[1, 2, 3, 4, 5, 6, 7, 8, 9] | val=[10, 11] | test=[12, 13]
Fold 1: train=[4, 5, 6, 7, 8, 9, 10, 11, 12] | val=[1, 13] | test=[2, 3] <== active
Fold 2: train=[1, 6, 7, 8, 9, 10, 11, 12, 13] | val=[2, 3] | test=[4, 5]
Fold 3: train=[1, 2, 3, 4, 9, 10, 11, 12, 13] | val=[5, 6] | test=[7, 8]
Fold 4: train=[1, 2, 3, 4, 5, 6, 11, 12, 13] | val=[7, 8] | test=[9, 10]
Train trajectory IDs: [ 4  5  6  7  8  9 10 11 12]
Val trajectory IDs  : [ 1 13]
Test trajectory IDs : [2 3]
Train samples: 3600
Val samples  : 800
Test samples : 800
Interpolated dvl_deeponet feature shape: (5200, 10)
Feature mean shape: (1, 10)
Feature std shape : (1, 10)
Velocity mean: [[ 1.9804777e+00 -8.6785639e-03  1.7561539e-03]]
Velocity std : [[0.43193492 0.30792665 0.07287853]]
LS sanity RMSE [m/s]: 0.18944333493709564
LS sanity mean error [m/s]: 0.1729569435119629


In [4]:
# ============================================================
# 2B. Raw 100 Hz IMU windows for the raw-IMU DeepONet branch
# ============================================================
# DVL stays at 1 Hz. IMU is kept as a 100-sample high-rate window.
# Output tensor:
#   imu_raw_windows_n: (N, 6, RAW_IMU_WINDOW)
# channels: [ax, ay, az, gx, gy, gz]
# ============================================================

RAW_IMU_WINDOW = 200
RAW_IMU_CHANNELS =6


def build_raw_imu_windows(t, traj_id, window_size=100):
    t_flat = np.asarray(t, dtype=np.float64).reshape(-1)
    traj_flat = np.asarray(traj_id).reshape(-1)
    if len(t_flat) != len(traj_flat):
        raise ValueError("t and traj_id must have the same length")

    windows = np.zeros((len(t_flat), RAW_IMU_CHANNELS, window_size), dtype=np.float32)

    for current_traj in np.unique(traj_flat):
        global_indices = np.flatnonzero(traj_flat == current_traj)
        if len(global_indices) == 0:
            continue

        traj_num = int(current_traj)
        imu_path = find_imu_path(traj_num)
        raw_imu = np.load(imu_path).astype(np.float32)
        if raw_imu.ndim != 2 or raw_imu.shape[1] != 7:
            raise ValueError(f"{imu_path} must have shape (M,7) = [time, ax, ay, az, gx, gy, gz], got {raw_imu.shape}")
        if not np.isfinite(raw_imu).all():
            raise ValueError(f"Non-finite values found in {imu_path}")

        imu_time = raw_imu[:, 0].astype(np.float64)
        imu_values = raw_imu[:, 1:7].astype(np.float32)
        if np.any(np.diff(imu_time) <= 0):
            raise ValueError(f"IMU timestamps are not strictly increasing in {imu_path}")

        imu_time_rel = imu_time - imu_time[0]
        dvl_time_traj = t_flat[global_indices]
        dvl_time_rel = dvl_time_traj - dvl_time_traj[0]

        for local_k, global_k in enumerate(global_indices):
            end_idx = np.searchsorted(imu_time_rel, dvl_time_rel[local_k], side="right")
            end_idx = max(end_idx, 1)
            start_idx = end_idx - window_size

            if start_idx >= 0:
                window = imu_values[start_idx:end_idx]
            else:
                available = imu_values[:end_idx]
                pad_count = window_size - len(available)
                padding = np.repeat(available[0:1], pad_count, axis=0)
                window = np.concatenate([padding, available], axis=0)

            if window.shape != (window_size, RAW_IMU_CHANNELS):
                raise RuntimeError(
                    f"Invalid raw IMU window shape at Trajectory{traj_num}, "
                    f"sample {local_k}: {window.shape}"
                )
            windows[global_k] = window.T

        print(
            f"Raw IMU windows built for Trajectory{traj_num}: "
            f"{len(global_indices)} DVL samples | raw IMU samples={len(raw_imu)}"
        )

    return windows


imu_raw_windows = build_raw_imu_windows(t, traj_id, window_size=RAW_IMU_WINDOW)

# Train-only normalization. This avoids test-fold leakage.
imu_raw_mean = imu_raw_windows[idx_train].mean(axis=(0, 2), keepdims=True).astype(np.float32)
imu_raw_std = np.maximum(
    imu_raw_windows[idx_train].std(axis=(0, 2), keepdims=True),
    1e-8,
).astype(np.float32)
imu_raw_windows_n = ((imu_raw_windows - imu_raw_mean) / imu_raw_std).astype(np.float32)

np.save(DATA_DIR / f"akit_imu_raw_100hz_windows_fold{CV_FOLD}.npy", imu_raw_windows)
np.save(DATA_DIR / f"akit_imu_raw_100hz_windows_normalized_fold{CV_FOLD}.npy", imu_raw_windows_n)

print("Raw IMU branch input shape:", imu_raw_windows_n.shape)
print("Raw IMU branch channels    :", IMU_FEATURE_NAMES)
print("Raw IMU train mean shape   :", imu_raw_mean.shape)
print("Raw IMU train std shape    :", imu_raw_std.shape)


Raw IMU windows built for Trajectory1: 400 DVL samples | raw IMU samples=40000
Raw IMU windows built for Trajectory2: 400 DVL samples | raw IMU samples=40000
Raw IMU windows built for Trajectory3: 400 DVL samples | raw IMU samples=40000
Raw IMU windows built for Trajectory4: 400 DVL samples | raw IMU samples=40000
Raw IMU windows built for Trajectory5: 400 DVL samples | raw IMU samples=40000
Raw IMU windows built for Trajectory6: 400 DVL samples | raw IMU samples=40000
Raw IMU windows built for Trajectory7: 400 DVL samples | raw IMU samples=40000
Raw IMU windows built for Trajectory8: 400 DVL samples | raw IMU samples=40000
Raw IMU windows built for Trajectory9: 400 DVL samples | raw IMU samples=40000
Raw IMU windows built for Trajectory10: 400 DVL samples | raw IMU samples=40000
Raw IMU windows built for Trajectory11: 400 DVL samples | raw IMU samples=40000
Raw IMU windows built for Trajectory12: 400 DVL samples | raw IMU samples=40000
Raw IMU windows built for Trajectory13: 400 DVL s

In [5]:
# ============================================================
# 3. Dataset
# ============================================================

class DVLIMUCartesianVelocityDataset(Dataset):
    def __init__(self, indices, window):
        self.window = int(window)
        indices = np.asarray(indices, dtype=np.int64)

        valid = []
        for i in indices:
            s = i - self.window + 1
            if s < 0:
                continue
            if np.all(traj_id[s:i + 1, 0] == traj_id[i, 0]):
                valid.append(i)
        self.valid_indices = np.asarray(valid, dtype=np.int64)
        if len(self.valid_indices) == 0:
            raise ValueError("No valid samples for this split; reduce WINDOW or adjust the split")

        self.features_n = torch.from_numpy(features_n.astype(np.float32))
        self.imu_raw_windows_n = torch.from_numpy(imu_raw_windows_n.astype(np.float32))
        self.t_n = torch.from_numpy(t_n.astype(np.float32))
        self.vel_gt_n = torch.from_numpy(vel_gt_n.astype(np.float32))
        self.beams_raw = torch.from_numpy(dvl_beams.astype(np.float32))
        self.t_raw = torch.from_numpy(t.astype(np.float32))
        self.traj_id = torch.from_numpy(traj_id.astype(np.int64))

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, item):
        i = int(self.valid_indices[item])
        s = i - self.window + 1
        e = i + 1
        return {
            "branch": self.features_n[s:e],               # DVL beams + averaged IMU baseline input
            "dvl_branch": self.features_n[s:e, :4],       # DVL beams only for raw-IMU branch model
            "raw_imu": self.imu_raw_windows_n[i],         # 100 Hz IMU, shape (6, 100)
            "trunk": self.t_n[i],
            "vel": self.vel_gt_n[i],
            "beam_raw": self.beams_raw[i],
            "t_raw": self.t_raw[i],
            "traj_id": self.traj_id[i, 0],
            "index": torch.tensor(i, dtype=torch.long),
        }


BATCH_SIZE = 128
NUM_WORKERS = 0

generator = torch.Generator()
generator.manual_seed(SEED)
loader_kwargs = dict(num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

train_ds = DVLIMUCartesianVelocityDataset(idx_train, WINDOW)
val_ds = DVLIMUCartesianVelocityDataset(idx_val, WINDOW)
test_ds = DVLIMUCartesianVelocityDataset(idx_test, WINDOW)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=generator, **loader_kwargs)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_kwargs)

print("Train dataset:", len(train_ds))
print("Val dataset  :", len(val_ds))
print("Test dataset :", len(test_ds))

sample = train_ds[0]
for key, value in sample.items():
    print(key, value.shape if hasattr(value, "shape") else value)


Train dataset: 3591
Val dataset  : 798
Test dataset : 798
branch torch.Size([2, 10])
dvl_branch torch.Size([2, 4])
raw_imu torch.Size([6, 200])
trunk torch.Size([1])
vel torch.Size([3])
beam_raw torch.Size([4])
t_raw torch.Size([1])
traj_id torch.Size([])
index torch.Size([])


In [6]:
# ============================================================
# 4. DVL-DeepONet models
# ============================================================
# Two completely independent model definitions:
#

#
# Model 2: Raw-IMU DVL-DeepONet
#          DVL branch + separate raw 100-Hz IMU branch
#
# IMPORTANT:
# The two models DO NOT share BranchCNN, TrunkMLP,
# prediction-head classes, or trainable parameters.
# ============================================================

USE_MULTIBRANCH_DEEPONET = True
MULTIBRANCH_TRAIN_CORRUPTION_PROB = 0.60

MODEL_VARIANTS_TO_RUN = [
    "avg",
    "raw_imu",
]


# ============================================================
# Missing-beam modes
# ============================================================

MULTIBRANCH_MISSING_PATTERNS = [
    (),        # mode 0: full beams

    (0,),      # mode 1
    (1,),      # mode 2
    (2,),      # mode 3
    (3,),      # mode 4

    (0, 1),    # mode 5
    (0, 2),    # mode 6
    (0, 3),    # mode 7
    (1, 2),    # mode 8
    (1, 3),    # mode 9
    (2, 3),    # mode 10

    (0, 1, 2), # mode 11
    (0, 1, 3), # mode 12
    (0, 2, 3), # mode 13
    (1, 2, 3), # mode 14
]


PATTERN_TO_BRANCH_MODE = {
    tuple(pattern): i
    for i, pattern in enumerate(
        MULTIBRANCH_MISSING_PATTERNS
    )
}

In [8]:
# ============================================================
# MODEL 2
# Raw-IMU DVL-DeepONet
# ============================================================

LATENT=32

class RawModelDVLBranchCNN(nn.Module):

    def __init__(
        self,
        window,
        latent_dim=LATENT
    ):
        super().__init__()

        self.net = nn.Sequential(

            # Only 4 DVL beams
            nn.Conv1d(
                4,
                32,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm1d(32),

            nn.SiLU(),

            nn.Conv1d(
                32,
                64,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm1d(64),

            nn.SiLU(),

            nn.Conv1d(
                64,
                64,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm1d(64),

            nn.SiLU(),

            nn.Flatten(),

            nn.Linear(
                64 * window,
                latent_dim
            ),

            nn.SiLU(),
        )


    def forward(self, x):

        x = x.permute(
            0,
            2,
            1
        )

        return self.net(x)


class RawModelIMUBranchCNN(nn.Module):

    def __init__(
        self,
        input_channels=6,
        latent_dim=LATENT
    ):
        super().__init__()

        self.net = nn.Sequential(

            nn.Conv1d(
                input_channels,
                32,
                kernel_size=5,
                padding=2
            ),

            nn.BatchNorm1d(32),

            nn.SiLU(),

            nn.Conv1d(
                32,
                64,
                kernel_size=5,
                stride=2,
                padding=2
            ),

            nn.BatchNorm1d(64),

            nn.SiLU(),

            nn.Conv1d(
                64,
                96,
                kernel_size=5,
                stride=2,
                padding=2
            ),

            nn.BatchNorm1d(96),

            nn.SiLU(),

            nn.AdaptiveAvgPool1d(1),

            nn.Flatten(),

            nn.Linear(
                96,
                latent_dim
            ),

            nn.SiLU(),
        )


    def forward(self, raw_imu):

        return self.net(
            raw_imu
        )

class RawModelTrunkMLP(nn.Module):

    def __init__(
        self,
        latent_dim=LATENT
    ):
        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                1,
                64
            ),

            nn.Tanh(),

            nn.Linear(
                64,
                128
            ),

            nn.Tanh(),

            nn.Linear(
                128,
                latent_dim
            ),

            nn.Tanh(),
        )


    def forward(self, t):

        return self.net(t)

class RawIMUDVLDeepONet(nn.Module):

    def __init__(
        self,
        window,
        vel_dim,
        latent_dim=LATENT,
        num_branch_modes=15
    ):
        super().__init__()

        self.vel_dim = vel_dim

        self.num_branch_modes = (
            num_branch_modes
        )


        # ----------------------------------------
        # DVL branches
        # ----------------------------------------

        self.dvl_branches = nn.ModuleList([

            RawModelDVLBranchCNN(
                window=window,
                latent_dim=latent_dim
            )

            for _ in range(
                num_branch_modes
            )
        ])


        # ----------------------------------------
        # Raw 100-Hz IMU branch
        # ----------------------------------------

        self.raw_imu_branch = (
            RawModelIMUBranchCNN(
                input_channels=RAW_IMU_CHANNELS,
                latent_dim=latent_dim
            )
        )


        # ----------------------------------------
        # Independent trunk
        # ----------------------------------------

        self.trunk = (
            RawModelTrunkMLP(
                latent_dim=latent_dim
            )
        )


        # ----------------------------------------
        # DVL + IMU fusion
        # ----------------------------------------

        self.fuse = nn.Sequential(

            nn.LayerNorm(
                latent_dim * 2
            ),

            nn.Linear(
                latent_dim * 2,
                latent_dim
            ),

            nn.SiLU(),
        )


        # ----------------------------------------
        # Independent prediction head
        # ----------------------------------------

        self.head = nn.Sequential(

            nn.LayerNorm(
                latent_dim
            ),

            nn.Linear(
                latent_dim,
                128
            ),

            nn.SiLU(),

            nn.Dropout(
                0.1
            ),

            nn.Linear(
                128,
                64
            ),

            nn.SiLU(),

            nn.Linear(
                64,
                vel_dim
            ),
        )


    def forward(
        self,
        branch,
        trunk,
        branch_mode=None,
        raw_imu=None,
        dvl_branch=None
    ):

        if raw_imu is None:

            raise ValueError(
                "RawIMUDVLDeepONet requires raw_imu."
            )


        if dvl_branch is None:

            dvl_branch = (
                branch[:, :, :4]
            )


        if branch_mode is None:

            branch_mode = torch.zeros(
                dvl_branch.shape[0],
                dtype=torch.long,
                device=dvl_branch.device
            )

        else:

            branch_mode = (
                branch_mode
                .to(dvl_branch.device)
                .long()
            )


        latent_dim = (
            self.dvl_branches[0]
            .net[-2]
            .out_features
        )


        dvl_feat = (
            dvl_branch.new_zeros(
                (
                    dvl_branch.shape[0],
                    latent_dim
                )
            )
        )


        for mode_id, branch_net in enumerate(
            self.dvl_branches
        ):

            mask = (
                branch_mode
                == mode_id
            )

            if torch.any(mask):

                dvl_feat[mask] = (
                    branch_net(
                        dvl_branch[mask]
                    )
                )


        # Raw IMU feature
        imu_feat = (
            self.raw_imu_branch(
                raw_imu
            )
        )


        # DVL + IMU fusion
        branch_feat = self.fuse(

            torch.cat(
                [
                    dvl_feat,
                    imu_feat
                ],
                dim=-1
            )
        )


        # Independent time trunk
        trunk_feat = (
            self.trunk(
                trunk
            )
        )


        # DeepONet interaction
        z = (
            branch_feat
            * trunk_feat
        )


        return self.head(z)        

In [10]:
# ============================================================
# 5. Clean loss utilities: MSE, Huber, Cauchy, and NAViLoss
# ============================================================

LOSS_MODES_TO_RUN = [
    "mse",
    "naviloss",
]

LOSS_DISPLAY_NAMES = {
    "mse": "DVL-DeepONet (MSE)",
    "naviloss": "DVL-DeepONet (NAViLoss, ours)",
}

# Training settings for every loss mode
EPOCHS = 100
LR = 1e-3
PATIENCE = 40
MIN_DELTA = 1e-6
LAMBDA_VEL = 1.0
LAMBDA_BEAM = 0.1

# NAViLoss parameters
NAVI_LAMBDA = 0.1
NAVI_EPS = 1e-8
NAVI_SIGMA_V = 1.0

# Robust baseline parameters
HUBER_DELTA = 1.0
CAUCHY_C = 1.0



DUP_LAMBDA = 0.1
DUP_ALPHA  = 0.1
DUP_BETA   = 0.01
DUP_MU     = 0.1
DUP_EPS    = 1e-8




vel_mean_t = torch.as_tensor(vel_mean, dtype=torch.float32, device=DEVICE)
vel_std_t  = torch.as_tensor(vel_std, dtype=torch.float32, device=DEVICE)
H_t = torch.as_tensor(H_np, dtype=torch.float32, device=DEVICE)


def denorm_vel(vel_hat_n):
    return vel_hat_n * vel_std_t + vel_mean_t


def to_device(batch):
    return {key: value.to(DEVICE, non_blocking=PIN_MEMORY) for key, value in batch.items()}


def pattern_to_active_mask(pattern, n_beams=4):
    mask = torch.ones(n_beams, dtype=torch.float32, device=DEVICE)
    for beam_idx in tuple(pattern):
        mask[int(beam_idx)] = 0.0
    return mask


def active_mask_from_branch_mode(branch_mode, batch_size, n_beams=4):
    if branch_mode is None:
        return torch.ones(batch_size, n_beams, dtype=torch.float32, device=DEVICE)
    branch_mode = branch_mode.detach().long().view(-1).to(DEVICE)
    masks = []
    for mode_id in branch_mode:
        pattern = MULTIBRANCH_MISSING_PATTERNS[int(mode_id.item())]
        masks.append(pattern_to_active_mask(pattern, n_beams=n_beams))
    return torch.stack(masks, dim=0)


def beam_condition_sigma(H_t, batch_size, device, active_mask=None, eps=1e-6):
    """
    Model-based DVL beam uncertainty from active beam geometry.
    Larger sigma means lower confidence. Shape: [batch_size, 4].
    """
    H_t = H_t.to(device)
    if active_mask is None:
        active_mask = torch.ones(batch_size, H_t.shape[0], device=device)
    else:
        active_mask = active_mask.to(device).float()

    sigmas = []
    for k in range(batch_size):
        H_active = H_t[active_mask[k] > 0.5]
        if H_active.shape[0] < 3:
            unc = torch.tensor(10.0, device=device)
        else:
            info = H_active.T @ H_active
            eigvals = torch.linalg.eigvalsh(info)
            unc = 1.0 / torch.clamp(eigvals[0], min=eps)
        sigmas.append(unc * torch.ones(H_t.shape[0], device=device))

    sigma = torch.stack(sigmas, dim=0)
    sigma = sigma / (sigma.mean() + eps)
    sigma = sigma + eps
    return sigma


def format_metric_value(value, decimals=3):
    if isinstance(value, list):
        return "[" + ", ".join(f"{x:.{decimals}f}" for x in value) + "]"
    return f"{value:.{decimals}f}"


def compute_velocity_metrics(pred, gt):
    err_vec = pred - gt
    err_norm = np.linalg.norm(err_vec, axis=1)
    per_axis_rmse = np.sqrt(np.mean(err_vec ** 2, axis=0))

    r2_axes = []
    vaf_axes = []
    for i in range(gt.shape[1]):
        y_true = gt[:, i]
        y_pred = pred[:, i]
        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
        r2 = 1.0 - ss_res / (ss_tot + 1e-12)
        vaf = (1.0 - np.var(y_true - y_pred) / (np.var(y_true) + 1e-12)) * 100.0
        r2_axes.append(float(r2))
        vaf_axes.append(float(vaf))

    return {
        "RMSE_vector": float(np.sqrt(np.mean(np.sum(err_vec ** 2, axis=1)))),
        "MAE_vector": float(np.mean(err_norm)),
        "STD_error_vector": float(np.std(err_norm)),
        "Median_error_vector": float(np.median(err_norm)),
        "Max_error_vector": float(np.max(err_norm)),
        "RMSE_axes": [float(x) for x in per_axis_rmse],
        "R2_axes": r2_axes,
        "R2_mean": float(np.mean(r2_axes)),
        "VAF_axes_percent": vaf_axes,
        "VAF_mean_percent": float(np.mean(vaf_axes)),
    }

def naviloss_term(r, sigma):
    return (1.0 / NAVI_LAMBDA) * (
        1.0 - 1.0 / (1.0 + NAVI_LAMBDA * (r**2 / (sigma**2 + NAVI_EPS)))
    )


def huber_term(r, delta=1.0):
    abs_r = torch.abs(r)
    return torch.where(abs_r <= delta, 0.5 * r**2, delta * (abs_r - 0.5 * delta))


def cauchy_term(r, c=1.0):
    return c**2 * torch.log1p((r / c) ** 2)


def velocity_loss_by_mode(vel_hat_n, vel_gt_n_batch, loss_mode):
    r = vel_hat_n - vel_gt_n_batch

    if loss_mode == "mse":
        return torch.mean(r**2)
    if loss_mode == "huber":
        return torch.mean(huber_term(r, HUBER_DELTA))
    if loss_mode == "cauchy":
        return torch.mean(cauchy_term(r, CAUCHY_C))
    if loss_mode == "naviloss":
        sigma_v = torch.full_like(r, float(NAVI_SIGMA_V))
        return torch.mean(naviloss_term(r, sigma_v))

    raise ValueError(f"Unknown loss_mode: {loss_mode}")


def beam_loss_by_mode(vel_hat_n, beam_raw, loss_mode, branch_mode=None):
    vel_hat = denorm_vel(vel_hat_n)
    vel_3d = vel_hat[:, :3]
    beam_pred = vel_3d @ H_t.T
    r = beam_pred - beam_raw

    if loss_mode == "mse":
        return torch.mean(r**2)

    if loss_mode == "naviloss":
        active_mask = active_mask_from_branch_mode(branch_mode, r.shape[0], n_beams=H_t.shape[0])
        sigma_b = beam_condition_sigma(H_t, batch_size=r.shape[0], device=r.device, active_mask=active_mask)
        return torch.mean(naviloss_term(r, sigma_b))

    
    raise ValueError(f"Unknown loss_mode: {loss_mode}")

print("Loss modes to run:", LOSS_MODES_TO_RUN)
print("NAViLoss lambda:", NAVI_LAMBDA)
print("NAViLoss sigma_v:", NAVI_SIGMA_V)
print("Beam uncertainty: condition-based active DVL geometry")


Loss modes to run: ['mse', 'huber', 'cauchy', 'naviloss']
NAViLoss lambda: 0.1
NAViLoss sigma_v: 1.0
Beam uncertainty: condition-based active DVL geometry


In [11]:
# ============================================================
# 6. Training/evaluation functions for MSE/Huber/Cauchy/NAViLoss
# ============================================================

MODEL_VARIANT_DISPLAY_NAMES = {
    "avg": "DVL-DeepONet averaged IMU",
    "raw_imu": "Navi-DeepONet",
}



def make_fresh_model(seed=SEED, model_variant="avg"):
    #set_seed(seed)
    set_seed(seed=SEED, deterministic=False)
    if model_variant == "avg":
        return AvgIMUMultiBranchDVLDeepONet(
            window=WINDOW,
            input_dim=INPUT_DIM,
            vel_dim=VEL_DIM,
            latent_dim=128,
            num_branch_modes=len(MULTIBRANCH_MISSING_PATTERNS),
        ).to(DEVICE)

    elif model_variant == "raw_imu":
        return RawIMUDVLDeepONet(
            window=WINDOW,
            vel_dim=VEL_DIM,
            latent_dim=128,
            num_branch_modes=len(MULTIBRANCH_MISSING_PATTERNS),
        ).to(DEVICE)

    else:
        raise ValueError(
            f"Unknown model_variant: {model_variant}"
        )





def make_multibranch_training_batch(branch, dvl_branch=None):
    if (not USE_MULTIBRANCH_DEEPONET) or MULTIBRANCH_TRAIN_CORRUPTION_PROB <= 0:
        branch_mode = torch.zeros(branch.shape[0], dtype=torch.long, device=branch.device)
        return branch, dvl_branch, branch_mode

    branch_aug = branch.clone()
    dvl_branch_aug = None if dvl_branch is None else dvl_branch.clone()
    branch_mode = torch.zeros(branch.shape[0], dtype=torch.long, device=branch.device)

    for i in range(branch.shape[0]):
        if torch.rand((), device=branch.device) < MULTIBRANCH_TRAIN_CORRUPTION_PROB:
            mode_id = int(torch.randint(1, len(MULTIBRANCH_MISSING_PATTERNS), (1,), device=branch.device).item())
            missing_pattern = MULTIBRANCH_MISSING_PATTERNS[mode_id]
            for beam_idx in missing_pattern:
                # Features are normalized. train_mean fill means normalized value 0.
                branch_aug[i, :, beam_idx] = 0.0
                if dvl_branch_aug is not None:
                    dvl_branch_aug[i, :, beam_idx] = 0.0
            branch_mode[i] = mode_id

    return branch_aug, dvl_branch_aug, branch_mode


def run_epoch_for_model(m, loader, optimizer, loss_mode, train=True, model_variant="avg"):
    m.train(train)
    sums = {"total": 0.0, "vel": 0.0, "beam": 0.0}
    count = 0

    for batch in loader:
        batch = to_device(batch)
        branch = batch["branch"]
        dvl_branch = batch.get("dvl_branch", None)
        raw_imu = batch.get("raw_imu", None)
        trunk = batch["trunk"]
        vel_gt_n_batch = batch["vel"]
        beam_raw = batch["beam_raw"]
        branch_mode = batch.get("branch_mode", None)

        if train:
            branch, dvl_branch, branch_mode = make_multibranch_training_batch(branch, dvl_branch=dvl_branch)
        elif branch_mode is None:
            branch_mode = torch.zeros(branch.shape[0], dtype=torch.long, device=branch.device)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            vel_hat_n = m(
                branch,
                trunk,
                branch_mode,
                raw_imu=raw_imu if model_variant == "raw_imu" else None,
                dvl_branch=dvl_branch,
            )
            L_vel = velocity_loss_by_mode(vel_hat_n, vel_gt_n_batch, loss_mode)
            L_beam = beam_loss_by_mode(vel_hat_n, beam_raw, loss_mode, branch_mode=branch_mode)
            L_total = LAMBDA_VEL * L_vel + LAMBDA_BEAM * L_beam

            if train:
                L_total.backward()
                torch.nn.utils.clip_grad_norm_(m.parameters(), max_norm=5.0)
                optimizer.step()

        bs = branch.shape[0]
        sums["total"] += float(L_total.detach().cpu()) * bs
        sums["vel"] += float(L_vel.detach().cpu()) * bs
        sums["beam"] += float(L_beam.detach().cpu()) * bs
        count += bs

    return {key: value / max(count, 1) for key, value in sums.items()}


@torch.no_grad()
def predict_dataset_model(m, dataset, batch_size=512, model_variant="avg"):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, **loader_kwargs)
    m.eval()
    pred_vel_list, t_list, idx_list, traj_list = [], [], [], []

    for batch in loader:
        batch = to_device(batch)
        vel_hat_n = m(
            batch["branch"],
            batch["trunk"],
            batch.get("branch_mode", None),
            raw_imu=batch.get("raw_imu", None) if model_variant == "raw_imu" else None,
            dvl_branch=batch.get("dvl_branch", None),
        )
        pred_vel_list.append(denorm_vel(vel_hat_n).cpu().numpy())
        t_list.append(batch["t_raw"].cpu().numpy())
        idx_list.append(batch["index"].cpu().numpy())
        traj_list.append(batch["traj_id"].cpu().numpy())

    pred_vel = np.concatenate(pred_vel_list, axis=0)
    t_out = np.concatenate(t_list, axis=0)[:, 0]
    idx_out = np.concatenate(idx_list, axis=0)
    traj_out = np.concatenate(traj_list, axis=0)
    return pred_vel, t_out, idx_out, traj_out


def train_one_loss_mode(loss_mode, seed=42, model_variant="avg"):
    display_model = MODEL_VARIANT_DISPLAY_NAMES.get(model_variant, model_variant)
    print("\n" + "=" * 80)
    print(f"Training model: {display_model} | loss: {loss_mode} | {LOSS_DISPLAY_NAMES.get(loss_mode, loss_mode)}")
    print("=" * 80)

    m = make_fresh_model(seed=seed, model_variant=model_variant)
    optimizer = torch.optim.AdamW(m.parameters(), lr=LR, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=10, factor=0.5
    )

    history = {
        "train_total": [], "val_total": [],
        "train_vel": [], "val_vel": [],
        "train_beam": [], "val_beam": [],
    }

    best_val = float("inf")
    best_state = None
    epochs_without_improvement = 0

    for epoch in range(1, EPOCHS + 1):
        train_m = run_epoch_for_model(m, train_loader, optimizer, loss_mode, train=True, model_variant=model_variant)
        val_m = run_epoch_for_model(m, val_loader, optimizer, loss_mode, train=False, model_variant=model_variant)
        scheduler.step(val_m["total"])

        for key in ["total", "vel", "beam"]:
            history[f"train_{key}"].append(train_m[key])
            history[f"val_{key}"].append(val_m[key])

        improved = val_m["total"] < best_val - MIN_DELTA
        if improved:
            best_val = val_m["total"]
            best_state = {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epoch == 1 or epoch % 10 == 0 or improved:
            print(
                f"[{model_variant}/{loss_mode}] Epoch {epoch:04d} | "
                f"Train={train_m['total']:.4e} | Val={val_m['total']:.4e} | "
                f"Vel={val_m['vel']:.2e} | Beam={val_m['beam']:.2e} | "
                f"LR={optimizer.param_groups[0]['lr']:.1e}"
            )

        if epochs_without_improvement >= PATIENCE:
            print(f"[{model_variant}/{loss_mode}] Early stopping at epoch {epoch}; best validation loss was {best_val:.4e}.")
            break

    if best_state is not None:
        m.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    ckpt_path = CKPT_DIR / f"best_dvl_deeponet_{model_variant}_{loss_mode}_fold{CV_FOLD}.pt"
    torch.save({
        "model_variant": model_variant,
        "loss_mode": loss_mode,
        "model_state": m.state_dict(),
        "history": history,
        "best_val": best_val,
        "config": {
            "seed": seed,
            "fold": CV_FOLD,
            "window": WINDOW,
            "raw_imu_window": RAW_IMU_WINDOW if model_variant == "raw_imu" else None,
            "input_dim": INPUT_DIM,
            "vel_dim": VEL_DIM,
            "multibranch_train_corruption_prob": MULTIBRANCH_TRAIN_CORRUPTION_PROB,
            "LAMBDA_VEL": LAMBDA_VEL,
            "LAMBDA_BEAM": LAMBDA_BEAM,
            "NAVI_LAMBDA": NAVI_LAMBDA,
            "NAVI_SIGMA_V": NAVI_SIGMA_V,
        }
    }, ckpt_path)

    return m, history, best_val, ckpt_path


In [12]:
# ============================================================
# 7. Missing-beam robustness helpers for 1/2/3 missing beams
# ============================================================

MISSING_BEAM_FILL = "train_mean"  # options: train_mean or zero

MISSING_SCENARIOS = {
    "full_beams": [()],
    "missing_1_beam": [(0,), (1,), (2,), (3,)],
    "missing_2_beams": [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)],
    "missing_3_beams": [(0, 1, 2), (0, 1, 3), (0, 2, 3), (1, 2, 3)],
}


def branch_mode_for_missing_pattern(pattern):
    pattern = tuple(sorted(pattern))
    return PATTERN_TO_BRANCH_MODE.get(pattern, 0)


def build_features_for_missing_pattern(pattern):
    pattern = tuple(sorted(pattern))
    corrupted = raw_features.copy()
    branch_mode_by_index = np.zeros(len(raw_features), dtype=np.int64)

    if len(pattern) > 0:
        if MISSING_BEAM_FILL == "train_mean":
            for beam_idx in pattern:
                corrupted[:, beam_idx] = feature_mean[0, beam_idx]
        elif MISSING_BEAM_FILL == "zero":
            corrupted[:, list(pattern)] = 0.0
        else:
            raise ValueError(f"Unsupported MISSING_BEAM_FILL: {MISSING_BEAM_FILL}")
        branch_mode_by_index[:] = branch_mode_for_missing_pattern(pattern)

    corrupted_n = (corrupted - feature_mean) / feature_std
    return corrupted, corrupted_n.astype(np.float32), branch_mode_by_index


class MissingBeamDataset(Dataset):
    def __init__(self, indices, window, features_override_n, branch_mode_by_index):
        self.window = int(window)
        self.features_override_n = torch.from_numpy(features_override_n.astype(np.float32))
        self.imu_raw_windows_n = torch.from_numpy(imu_raw_windows_n.astype(np.float32))
        self.branch_mode_by_index = np.asarray(branch_mode_by_index, dtype=np.int64)
        indices = np.asarray(indices, dtype=np.int64)

        valid = []
        for i in indices:
            s = i - self.window + 1
            if s < 0:
                continue
            if np.all(traj_id[s:i + 1, 0] == traj_id[i, 0]):
                valid.append(i)
        self.valid_indices = np.asarray(valid, dtype=np.int64)

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, item):
        i = int(self.valid_indices[item])
        s = i - self.window + 1
        e = i + 1
        return {
            "branch": self.features_override_n[s:e],
            "dvl_branch": self.features_override_n[s:e, :4],
            "raw_imu": self.imu_raw_windows_n[i],
            "trunk": torch.from_numpy(t_n[i].astype(np.float32)),
            "t_raw": torch.from_numpy(t[i].astype(np.float32)),
            "traj_id": torch.tensor(traj_id[i, 0], dtype=torch.long),
            "index": torch.tensor(i, dtype=torch.long),
            "branch_mode": torch.tensor(self.branch_mode_by_index[i], dtype=torch.long),
        }


@torch.no_grad()
def predict_missing_dataset_model(m, dataset, batch_size=512, model_variant="avg"):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, **loader_kwargs)
    m.eval()
    pred_vel_list, t_list, idx_list, traj_list, mode_list = [], [], [], [], []

    for batch in loader:
        batch = to_device(batch)
        vel_hat_n = m(
            batch["branch"],
            batch["trunk"],
            batch.get("branch_mode", None),
            raw_imu=batch.get("raw_imu", None) if model_variant == "raw_imu" else None,
            dvl_branch=batch.get("dvl_branch", None),
        )
        pred_vel_list.append(denorm_vel(vel_hat_n).cpu().numpy())
        t_list.append(batch["t_raw"].cpu().numpy())
        idx_list.append(batch["index"].cpu().numpy())
        traj_list.append(batch["traj_id"].cpu().numpy())
        mode_list.append(batch["branch_mode"].cpu().numpy())

    pred_vel = np.concatenate(pred_vel_list, axis=0)
    t_out = np.concatenate(t_list, axis=0).reshape(-1)
    idx_out = np.concatenate(idx_list, axis=0)
    traj_out = np.concatenate(traj_list, axis=0)
    mode_out = np.concatenate(mode_list, axis=0)
    return pred_vel, t_out, idx_out, traj_out, mode_out


def evaluate_model_on_pattern(m, pattern, indices=idx_test, model_variant="avg"):
    _, features_missing_n, branch_mode_by_index = build_features_for_missing_pattern(pattern)
    ds = MissingBeamDataset(indices, WINDOW, features_missing_n, branch_mode_by_index)
    pred, t_out, idx_out, traj_out, mode_out = predict_missing_dataset_model(m, ds, model_variant=model_variant)
    gt = dvl_cart_gt[idx_out, :VEL_DIM]
    return compute_velocity_metrics(pred, gt), pred, gt, idx_out


def evaluate_model_on_scenario(m, scenario_name, patterns=None, indices=idx_test, model_variant="avg"):
    if patterns is None:
        patterns = MISSING_SCENARIOS[scenario_name]
    preds, gts, pattern_metrics = [], [], {}
    for pattern in patterns:
        metrics, pred, gt, idx_out = evaluate_model_on_pattern(
            m,
            pattern,
            indices=indices,
            model_variant=model_variant,
        )
        pattern_key = "full" if len(pattern) == 0 else "missing_" + "_".join(str(p + 1) for p in pattern)
        pattern_metrics[pattern_key] = metrics
        preds.append(pred)
        gts.append(gt)
    pred_all = np.concatenate(preds, axis=0)
    gt_all = np.concatenate(gts, axis=0)
    avg_metrics = compute_velocity_metrics(pred_all, gt_all)
    return {"average": avg_metrics, "patterns": pattern_metrics}

print("Missing scenarios:")
for name, patterns in MISSING_SCENARIOS.items():
    print(f"  {name}: {patterns}")


Missing scenarios:
  full_beams: [()]
  missing_1_beam: [(0,), (1,), (2,), (3,)]
  missing_2_beams: [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
  missing_3_beams: [(0, 1, 2), (0, 1, 3), (0, 2, 3), (1, 2, 3)]


In [13]:
# ============================================================
# 8. Train selected neural models and evaluate corrupted-beam scenarios

# ============================================================



LOSS_MODES_BY_MODEL_VARIANT = {
   # "avg": ["mse"],
    "raw_imu": [ "naviloss"],
}

all_loss_results = {}
trained_models = {}
loss_histories = {}
loss_checkpoints = {}

for model_variant in MODEL_VARIANTS_TO_RUN:
    for loss_mode in LOSS_MODES_BY_MODEL_VARIANT.get(model_variant, []):
        result_key = f"{model_variant}_{loss_mode}"
        model_i, history_i, best_val_i, ckpt_i = train_one_loss_mode(
            loss_mode,
            seed=SEED,
            model_variant=model_variant,
        )
        trained_models[result_key] = model_i
        loss_histories[result_key] = history_i
        loss_checkpoints[result_key] = str(ckpt_i)

        # Clean/full-beam test prediction
        pred_test, t_test, idx_test_pred, traj_test_pred = predict_dataset_model(
            model_i,
            test_ds,
            model_variant=model_variant,
        )
        gt_test = dvl_cart_gt[idx_test_pred, :VEL_DIM]
        clean_metrics = compute_velocity_metrics(pred_test, gt_test)

        # Corrupted/missing-beam test scenarios
        scenario_results = {}
        for scenario_name in ["missing_1_beam", "missing_2_beams", "missing_3_beams"]:
            scenario_results[scenario_name] = evaluate_model_on_scenario(
                model_i,
                scenario_name,
                model_variant=model_variant,
            )

        model_display = MODEL_VARIANT_DISPLAY_NAMES.get(model_variant, model_variant)
        if loss_mode == "mse":
            loss_label = "MSE"
        elif loss_mode == "naviloss":
            loss_label = "NAViLoss"
        elif loss_mode == "huber":
            loss_label = "HuberLoss"
        else:
            loss_label = loss_mode.title()
        display_name = f"{model_display} ({loss_label})"

        all_loss_results[result_key] = {
            "model_variant": model_variant,
            "loss_mode": loss_mode,
            "display_name": display_name,
            "loss_display_name": LOSS_DISPLAY_NAMES.get(loss_mode, loss_mode),
            "best_val_loss": float(best_val_i),
            "checkpoint": str(ckpt_i),
            "clean_test": clean_metrics,
            "missing_scenarios": scenario_results,
        }

        print("\n========== RESULT:", display_name, "==========")
        print("Clean/full-beam test:")
        for key, value in clean_metrics.items():
            print(f"  {key}: {format_metric_value(value, decimals=3)}")
        for scenario_name, scenario_data in scenario_results.items():
            print(f"{scenario_name} average:")
            for key, value in scenario_data["average"].items():
                print(f"  {key}: {format_metric_value(value, decimals=3)}")

with open(SAVE_DIR / f"selected_corrupted_beam_neural_results_fold{CV_FOLD}.json", "w") as f:
    json.dump(all_loss_results, f, indent=2)

print("\nSaved:", SAVE_DIR / f"selected_corrupted_beam_neural_results_fold{CV_FOLD}.json")
print("Trained result keys:", list(all_loss_results.keys()))



Training model: DVL-DeepONet raw-IMU branch | loss: naviloss | DVL-DeepONet (NAViLoss, ours)
[raw_imu/naviloss] Epoch 0001 | Train=3.7635e-01 | Val=1.3274e-01 | Vel=1.32e-01 | Beam=4.86e-03 | LR=1.0e-03
[raw_imu/naviloss] Epoch 0002 | Train=2.2915e-01 | Val=7.9343e-02 | Vel=7.89e-02 | Beam=4.19e-03 | LR=1.0e-03
[raw_imu/naviloss] Epoch 0005 | Train=1.8334e-01 | Val=7.8684e-02 | Vel=7.82e-02 | Beam=4.34e-03 | LR=1.0e-03
[raw_imu/naviloss] Epoch 0010 | Train=1.3629e-01 | Val=1.0449e-01 | Vel=1.04e-01 | Beam=4.21e-03 | LR=1.0e-03
[raw_imu/naviloss] Epoch 0011 | Train=1.3216e-01 | Val=7.7566e-02 | Vel=7.72e-02 | Beam=4.08e-03 | LR=1.0e-03
[raw_imu/naviloss] Epoch 0012 | Train=1.2503e-01 | Val=7.6628e-02 | Vel=7.63e-02 | Beam=3.71e-03 | LR=1.0e-03
[raw_imu/naviloss] Epoch 0013 | Train=1.2495e-01 | Val=6.4713e-02 | Vel=6.43e-02 | Beam=3.79e-03 | LR=1.0e-03
[raw_imu/naviloss] Epoch 0020 | Train=1.1962e-01 | Val=6.8946e-02 | Vel=6.86e-02 | Beam=3.32e-03 | LR=1.0e-03
[raw_imu/naviloss] Epoch 0

In [14]:
# ============================================================
# 9. Save checkpoints, models, histories, predictions, results
#    RUN THIS AFTER SECTION 8
# ============================================================

import json
import pickle
import shutil
import numpy as np
import torch
from pathlib import Path


# ------------------------------------------------------------
# Create export folder
# ------------------------------------------------------------

EXPORT_DIR = SAVE_DIR / f"fold_{CV_FOLD}_final_exports_ARUP"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("Saving everything to:")
print(EXPORT_DIR)


# ------------------------------------------------------------
# Helper for JSON serialization
# ------------------------------------------------------------

def make_json_serializable(obj):

    if isinstance(obj, dict):
        return {
            str(k): make_json_serializable(v)
            for k, v in obj.items()
        }

    elif isinstance(obj, (list, tuple)):
        return [
            make_json_serializable(v)
            for v in obj
        ]

    elif isinstance(obj, np.ndarray):
        return obj.tolist()

    elif isinstance(obj, np.integer):
        return int(obj)

    elif isinstance(obj, np.floating):
        return float(obj)

    elif isinstance(obj, torch.Tensor):
        return obj.detach().cpu().numpy().tolist()

    elif isinstance(obj, Path):
        return str(obj)

    return obj


# ============================================================
# Save each trained model
# ============================================================

for result_key, model_i in trained_models.items():

    print("\n" + "=" * 70)
    print("SAVING:", result_key)
    print("=" * 70)

    model_dir = EXPORT_DIR / result_key
    model_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    # --------------------------------------------------------
    # Get experiment information
    # --------------------------------------------------------

    result_data = all_loss_results[result_key]

    model_variant = result_data["model_variant"]
    loss_mode = result_data["loss_mode"]

    history_i = loss_histories[result_key]

    original_ckpt = Path(
        loss_checkpoints[result_key]
    )


    # ========================================================
    # 1. COPY ORIGINAL BEST CHECKPOINT
    # ========================================================

    if original_ckpt.exists():

        suffix = original_ckpt.suffix

        if suffix == "":
            suffix = ".pt"

        copied_ckpt = (
            model_dir /
            f"best_training_checkpoint{suffix}"
        )

        shutil.copy2(
            original_ckpt,
            copied_ckpt
        )

        print(
            "Best checkpoint copied:",
            copied_ckpt
        )

    else:

        copied_ckpt = None

        print(
            "WARNING: checkpoint not found:",
            original_ckpt
        )


    # ========================================================
    # 2. SAVE CURRENT MODEL STATE_DICT
    # ========================================================

    model_state_path = (
        model_dir /
        "model_state_dict.pt"
    )

    torch.save(
        model_i.state_dict(),
        model_state_path
    )

    print(
        "Model state_dict saved:",
        model_state_path
    )


    # ========================================================
    # 3. SAVE TRAINING HISTORY
    # ========================================================

    history_pkl_path = (
        model_dir /
        "training_history.pkl"
    )

    with open(
        history_pkl_path,
        "wb"
    ) as f:

        pickle.dump(
            history_i,
            f
        )


    history_json_path = (
        model_dir /
        "training_history.json"
    )

    with open(
        history_json_path,
        "w"
    ) as f:

        json.dump(
            make_json_serializable(
                history_i
            ),
            f,
            indent=2
        )

    print(
        "Training history saved."
    )


    # ========================================================
    # 4. CLEAN TEST PREDICTIONS
    # ========================================================

    print(
        "Generating clean-test predictions..."
    )

    pred_test, \
    t_test, \
    idx_test_pred, \
    traj_test_pred = predict_dataset_model(
        model_i,
        test_ds,
        model_variant=model_variant,
    )


    pred_test = np.asarray(
        pred_test
    )

    t_test = np.asarray(
        t_test
    )

    idx_test_pred = np.asarray(
        idx_test_pred
    )

    traj_test_pred = np.asarray(
        traj_test_pred
    )


    gt_test = np.asarray(
        dvl_cart_gt[
            idx_test_pred,
            :VEL_DIM
        ]
    )


    # --------------------------------------------------------
    # Save prediction arrays
    # --------------------------------------------------------

    clean_prediction_path = (
        model_dir /
        "clean_test_predictions.npz"
    )

    np.savez_compressed(

        clean_prediction_path,

        prediction=pred_test,
        ground_truth=gt_test,

        time=t_test,

        indices=idx_test_pred,

        trajectory=traj_test_pred,
    )

    print(
        "Clean predictions saved:",
        clean_prediction_path
    )


    # ========================================================
    # 5. SAVE CLEAN METRICS
    # ========================================================

    clean_metrics = compute_velocity_metrics(
        pred_test,
        gt_test
    )


    with open(
        model_dir /
        "clean_test_metrics.json",
        "w"
    ) as f:

        json.dump(
            make_json_serializable(
                clean_metrics
            ),
            f,
            indent=2
        )


    # ========================================================
    # 6. SAVE ALREADY-COMPUTED MISSING-BEAM RESULTS
    # ========================================================

    scenario_results = result_data[
        "missing_scenarios"
    ]


    # Pickle version
    with open(
        model_dir /
        "missing_beam_results.pkl",
        "wb"
    ) as f:

        pickle.dump(
            scenario_results,
            f
        )


    # JSON version
    with open(
        model_dir /
        "missing_beam_results.json",
        "w"
    ) as f:

        json.dump(
            make_json_serializable(
                scenario_results
            ),
            f,
            indent=2
        )


    # ========================================================
    # 7. SAVE COMPLETE CHECKPOINT
    # ========================================================

    complete_checkpoint = {

        "model_state_dict":
            model_i.state_dict(),

        "model_variant":
            model_variant,

        "loss_mode":
            loss_mode,

        "result_key":
            result_key,

        "cv_fold":
            CV_FOLD,

        "seed":
            SEED,

        "best_val_loss":
            result_data[
                "best_val_loss"
            ],

        "history":
            history_i,

        "clean_test_metrics":
            clean_metrics,

        "missing_scenarios":
            scenario_results,

        "original_checkpoint":
            str(original_ckpt),
    }


    complete_checkpoint_path = (
        model_dir /
        "complete_checkpoint.pt"
    )

    torch.save(
        complete_checkpoint,
        complete_checkpoint_path
    )

    print(
        "Complete checkpoint saved:",
        complete_checkpoint_path
    )


    # ========================================================
    # 8. SAVE RUN INFORMATION
    # ========================================================

    run_info = {

        "result_key":
            result_key,

        "model_variant":
            model_variant,

        "loss_mode":
            loss_mode,

        "cv_fold":
            CV_FOLD,

        "seed":
            SEED,

        "best_val_loss":
            result_data[
                "best_val_loss"
            ],

        "original_checkpoint":
            str(original_ckpt),

        "copied_checkpoint":
            (
                str(copied_ckpt)
                if copied_ckpt
                is not None
                else None
            ),

        "model_state_dict":
            str(model_state_path),

        "complete_checkpoint":
            str(
                complete_checkpoint_path
            ),

        "clean_predictions":
            str(clean_prediction_path),
    }


    with open(
        model_dir /
        "run_info.json",
        "w"
    ) as f:

        json.dump(
            make_json_serializable(
                run_info
            ),
            f,
            indent=2
        )


# ============================================================
# 9. SAVE MASTER RESULT FILE
# ============================================================

master_path = (
    EXPORT_DIR /
    "all_loss_results.json"
)

with open(
    master_path,
    "w"
) as f:

    json.dump(
        make_json_serializable(
            all_loss_results
        ),
        f,
        indent=2
    )


# ============================================================
# DONE
# ============================================================

print("\n" + "=" * 75)
print("EVERYTHING SAVED SUCCESSFULLY")
print("=" * 75)

print("\nLocation:")
print(EXPORT_DIR)

print("\nSaved models:")

for result_key in trained_models.keys():
    print("  ", result_key)

print("\nFor each model you now have:")
print("  - original best checkpoint copy")
print("  - model_state_dict.pt")
print("  - complete_checkpoint.pt")
print("  - training_history.pkl")
print("  - training_history.json")
print("  - clean_test_predictions.npz")
print("  - clean_test_metrics.json")
print("  - missing_beam_results.pkl")
print("  - missing_beam_results.json")
print("  - run_info.json")

Saving everything to:
dvl_deeponet_100_imu_missing_beam_naviloss/fold_1_final_exports_ARUP

SAVING: raw_imu_naviloss
Best checkpoint copied: dvl_deeponet_100_imu_missing_beam_naviloss/fold_1_final_exports_ARUP/raw_imu_naviloss/best_training_checkpoint.pt
Model state_dict saved: dvl_deeponet_100_imu_missing_beam_naviloss/fold_1_final_exports_ARUP/raw_imu_naviloss/model_state_dict.pt
Training history saved.
Generating clean-test predictions...
Clean predictions saved: dvl_deeponet_100_imu_missing_beam_naviloss/fold_1_final_exports_ARUP/raw_imu_naviloss/clean_test_predictions.npz
Complete checkpoint saved: dvl_deeponet_100_imu_missing_beam_naviloss/fold_1_final_exports_ARUP/raw_imu_naviloss/complete_checkpoint.pt

EVERYTHING SAVED SUCCESSFULLY

Location:
dvl_deeponet_100_imu_missing_beam_naviloss/fold_1_final_exports_ARUP

Saved models:
   raw_imu_naviloss

For each model you now have:
  - original best checkpoint copy
  - model_state_dict.pt
  - complete_checkpoint.pt
  - training_histor